### Create database 'performance' for the performance indicator and populate it with tables and static records

The demo showcases the story
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-735


## 1 - Initialisation

In [ ]:
# Access to Prefect
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  

init_demo()

# Reload the global vars again
from resources.utils import *  

In [ ]:
from importlib import reload
import os
import sys
import prefect
from rs_common import prefect_utils
from rs_common.prefect_utils import *
import rs_workflows

# Local paths
rs_workflows_parent = Path(rs_workflows.__path__[0]).parent

In [ ]:
flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
      
  },
}
# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

## 2 - Deploy Prefect flows

We deploy the Prefect workflow that is implemented in the rs-client-libraries git repository to model the database, thus to create the tables and to insert the static records in the `pi_category` table

WARNING: the rs-client-libraries source code must be identical in these 3 environments:

- https://github.com/RS-PYTHON/rs-demo.git
- This Jupyter environment
- The Prefect Docker images

In [ ]:
%%bash -s "$rs_workflows_parent"
# Deploy the flow
deploy_file=$(realpath "./init_pi_db_flows.yaml")
echo "Deploying '$deploy_file'..."
(cd $1; prefect --no-prompt deploy --prefect-file "$deploy_file" --all)

In [ ]:
# Flow deployment names
pi_deploy = "initialize-pi-db/Initialize PI DB"
await prefect_utils.wait_for_deployment(pi_deploy)

## 3 - Run flow

Run the flow to init the pi database.

In [ ]:
# Convert to json to trigger prefect flow
params_str = to_json(flow_parameters)

In [ ]:
%%bash -s "$pi_deploy" "$params_str"
prefect deployment run "$1" --params "$2" --watch